# مختبر اليوم الثالث — تصنيف مغادرة العملاء وانحدار سعر المركبة

> هذا الدفتر مبني مباشرة من مواصفات مختبرات منافذ المعتمدة للدورة.


In [ ]:
from pathlib import Path

# يعمل محليًا داخل المستودع أو عند وضع الحزمة في مجلد مستقل
DATA_CANDIDATES = [Path("manafeth_data_package"), Path("data/raw"), Path("../../data/raw")]
DATA_DIR = next((p for p in DATA_CANDIDATES if p.exists()), DATA_CANDIDATES[0])
CUSTOMERS_PATH = DATA_DIR / "manafeth_customers.parquet"
ORDERS_PATH = DATA_DIR / "manafeth_orders.parquet"
VEHICLES_PATH = DATA_DIR / "markabat_listings_sample.csv"
SHIFTED_PATH = DATA_DIR / "shifted_month.parquet"
print("DATA_DIR:", DATA_DIR.resolve())


## هدف المختبر

ينفذ الطالب مهمتين مختلفتين على بيانات الحزمة نفسها: تصنيف مغادرة العميل، وانحدار سعر مركبة مستعملة. الهدف هو تثبيت الفرق بين **الفئة** و**الرقم ذي المقدار** لا تحقيق أفضل أداء ممكن.

## السيناريو والبيانات

يستخدم مسار التصنيف جدول العملاء وهدف `churned_30d`. ويستخدم مسار الانحدار ملف `markabat_listings_sample.csv` الذي يحتوي على 2,000 إعلان لمركبات مستعملة؛ الهدف هو `sale_price_sar`.

| المسار | السؤال | الملف | الهدف | نوع المهمة |
|---|---|---|---|---|
| التصنيف | هل سيغادر العميل خلال 30 يومًا؟ | `manafeth_customers.parquet` | `churned_30d` | تصنيف ثنائي |
| الانحدار | ما سعر البيع التقريبي للمركبة؟ | `markabat_listings_sample.csv` | `sale_price_sar` | انحدار |

## خطوات التنفيذ بالتسلسل

1. أعد استخدام `preprocessor` وخط تقسيم العملاء من اليوم الثاني.
2. أنشئ خط تصنيف يضم `preprocessor` ثم الانحدار اللوجستي.
3. درب النموذج على `X_train` و`y_train`، ثم احفظ الفئات المتوقعة واحتمال المغادرة.
4. افتح ملف المركبات وفحص الأعمدة.
5. استبعد `listing_id` من خصائص المركبة؛ وفي أول نموذج استبعد `listed_month` و`days_on_platform` لأن الهدف التعليمي هو التنبؤ بسعر عند إنشاء الإعلان لا بعد بقائه على المنصة.
6. ابنِ خط معالجة جديدًا للمركبات: أعمدة رقمية وفئوية، ثم انحدار خطي.
7. درب نموذج السعر واطبع أول خمسة أسعار متوقعة بعد التقريب إلى ريالين عشريين.
8. افحص صفًا واحدًا من كل مسار: خصائصه، التوقع، والحقيقة التي نحتفظ بها للتقييم في اليوم الرابع.

## ما يكتبه أو يشغله الطالب


In [ ]:
from sklearn.linear_model import LogisticRegression, LinearRegression

churn_classifier = Pipeline([
    ("prepare", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])
churn_classifier.fit(X_train, y_train)
churn_predictions = churn_classifier.predict(X_test)
churn_probabilities = churn_classifier.predict_proba(X_test)[:, 1]

vehicles = pd.read_csv(VEHICLES_PATH)
vehicle_target = "sale_price_sar"
vehicle_features = [
    "make", "model_year", "mileage_km", "city",
    "condition_grade", "photos_count"
]

X_vehicle = vehicles[vehicle_features]
y_vehicle = vehicles[vehicle_target]
Xv_train, Xv_test, yv_train, yv_test = train_test_split(
    X_vehicle, y_vehicle, test_size=0.20, random_state=42
)

vehicle_numeric = ["model_year", "mileage_km", "photos_count"]
vehicle_categorical = ["make", "city", "condition_grade"]
vehicle_preprocessor = ColumnTransformer([
    ("numbers", Pipeline([
        ("fill", SimpleImputer(strategy="median")),
        ("scale", StandardScaler())
    ]), vehicle_numeric),
    ("categories", Pipeline([
        ("fill", SimpleImputer(strategy="most_frequent")),
        ("encode", OneHotEncoder(handle_unknown="ignore"))
    ]), vehicle_categorical)
])

price_regressor = Pipeline([
    ("prepare", vehicle_preprocessor),
    ("model", LinearRegression())
])
price_regressor.fit(Xv_train, yv_train)
price_predictions = price_regressor.predict(Xv_test)
print(price_predictions[:5].round(2))


## النتيجة المتوقعة

يحصل الطالب على قائمة فئات واحتمالات لمغادرة العملاء، وقائمة أسعار رقمية متوقعة للمركبات. لا يعلن أن نموذجًا جيدًا من قراءة خمسة صفوف؛ سيكون قياس الجودة محور اليوم الرابع.

## المهارات التي يراجعها الطالب

يختار الطالب التصنيف أو الانحدار من شكل الهدف، ويضع نموذجًا داخل `Pipeline`، ويستخدم `fit` و`predict` و`predict_proba`، ويتحقق من أن المعرفات وأعمدة المستقبل لا تدخل الخصائص.

## شرط التسليم قبل مغادرة اليوم

يسلم الطالب لقطة ناتج أو خلية طباعة من المسارين، ثم يكتب جملتين: «مخرج التصنيف …» و«مخرج الانحدار …». يجب أن تتضمن الجملتان الفرق بين فئة واحتمال ورقم.

---
